## This is entirely for CUDA/Colab T4 run


In [1]:
%cd /content
!rm -rf /content/AttnResGPT-next-scale
!git clone https://github.com/AtinChing/AttnResGPT-next-scale.git /content/AttnResGPT-next-scale
%cd /content/AttnResGPT-next-scale
!pip install -q -r requirements.txt


/content
Cloning into '/content/AttnResGPT-next-scale'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 190 (delta 93), reused 165 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 1.17 MiB | 20.58 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/AttnResGPT-next-scale
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 114.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.0 MB/s eta 0:00:00


In [ ]:
import shutil
from pathlib import Path

REPO = Path('/content/AttnResGPT-next-scale')
DRIVE_ARTIFACT_FOLDER = 'AttnResGPT-next-scale-artifacts'
USE_GOOGLE_DRIVE = True


def _symlink_to_drive(repo: Path, name: str, target: Path) -> None:
    local = repo / name
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() == target.resolve():
            return
        local.unlink()
    elif local.exists():
        if local.is_file():
            local.unlink()
        else:
            for child in local.iterdir():
                if child.name.startswith('.'):
                    if child.is_dir():
                        shutil.rmtree(child)
                    else:
                        child.unlink()
                    continue
                dest = target / child.name
                if dest.exists():
                    shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
                shutil.move(str(child), str(dest))
            shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)
    print(f'linked {name} -> {target}')


if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive') / DRIVE_ARTIFACT_FOLDER
    for sub in ('checkpoints', 'outputs'):
        (drive_root / sub).mkdir(parents=True, exist_ok=True)
    print('VLM artifacts on Drive:', drive_root.resolve())
    _symlink_to_drive(REPO, 'checkpoints', drive_root / 'checkpoints')
    _symlink_to_drive(REPO, 'outputs', drive_root / 'outputs')
else:
    print('VLM artifacts on VM disk under', REPO)

In [2]:
import os
# os.environ['WANDB_API_KEY'] = '<your-key>'  # or Colab secrets


In [3]:
!nvidia-smi

Thu Mar 26 21:36:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
%cd /content/AttnResGPT-next-scale
!python experiments/vlm_attnres_flickr30k.py \
  --entity atin5551-uc-davis \
  --project attnres-next-scale \
  --run-name vlm_attnres_flickr30k_t4_2 \
  --vision-model google/siglip-base-patch16-224 \
  --dataset-name Mozilla/flickr30k-transformed-captions \
  --dataset-split train \
  --tokenizer-name gpt2 \
  --decoder-config configs/large_ctx512_3000.yaml \
  --decoder-backend gpt_attnres_large \
  --device cuda \
  --batch-size 8 \
  --max-examples 20000 \
  --max-text-tokens 128 \
  --num-workers 2 \
  --epochs 3 \
  --learning-rate 1e-4 \
  --weight-decay 0.01 \
  --warmup-steps 100 \
  --checkpoint-interval 500 \
  --plateau-patience 2 \
  --mixed-precision \
  --amp-dtype float16


/content/AttnResGPT-next-scale
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100% 408/408 [00:00<00:00, 502.98it/s, Materializing param=vision_model.post_layernorm.weight]                      
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: atin5551 (atin5551-uc-davis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ setting up run 3hwjyfm8 (0.3s)
wandb: ⣾ setting up run 3hwjyfm8 (0.3s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/AttnResGPT-next-scale/wandb/run-20260326_0655

In [4]:
%cd /content/AttnResGPT-next-scale
!python experiments/vlm_attnres_flickr30k.py \
  --entity atin5551-uc-davis \
  --project attnres-next-scale \
  --run-name vlm_baseline_flickr30k_t4_b8_2 \
  --vision-model google/siglip-base-patch16-224 \
  --dataset-name Mozilla/flickr30k-transformed-captions \
  --dataset-split train \
  --tokenizer-name gpt2 \
  --decoder-config configs/large_ctx512_3000.yaml \
  --decoder-architecture baseline \
  --decoder-backend gpt_baseline_large \
  --device cuda \
  --batch-size 8 \
  --max-examples 20000 \
  --max-text-tokens 128 \
  --num-workers 2 \
  --epochs 3 \
  --learning-rate 1e-4 \
  --weight-decay 0.01 \
  --warmup-steps 100 \
  --checkpoint-interval 500 \
  --plateau-patience 2 \
  --mixed-precision \
  --amp-dtype float16


/content/AttnResGPT-next-scale
config.json: 100% 665/665 [00:00<00:00, 2.46MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 99.9kB/s]
vocab.json: 1.04MB [00:00, 6.40MB/s]
merges.txt: 456kB [00:00, 9.38MB/s]
tokenizer.json: 1.36MB [00:00, 7.97MB/s]
preprocessor_config.json: 100% 368/368 [00:00<00:00, 1.81MB/s]
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
config.json: 100% 432/432 [00:00<00:00, 2.02MB/s]
tokenizer_config.json: 100% 711/711 [00:00<00:00, 3.93MB/s]
spiece.model: 100% 798k/798k [00:00<00:00, 967kB/s] 
special_tokens_map.json: 100% 409/409 [00:00<00:00, 2.35MB/s]
tokenizer.json: 2.40MB [00:00, 111MB/s]
model.safetensors: 100% 813M/813M [00:07<00:00, 116MB/s] 
Loading weights: 100% 408/408 [00:00<00:00,

In [ ]:
ENTITY = "atin5551-uc-davis"
PROJECT = "attnres-next-scale"
RUN_NAME = "vlm_attnres_flickr30k_t4"
import wandb
import glob

api = wandb.Api()
artifact_dir = None
artifact_name_used = None

for step in range(4000, 0, -500):
    artifact_name = f"{ENTITY}/{PROJECT}/{RUN_NAME}_step_{step:07d}:latest"
    try:
        art = api.artifact(artifact_name)
        artifact_dir = art.download(root="/content/wandb_restores")
        artifact_name_used = artifact_name
        print("Downloaded:", artifact_name)
        print("To:", artifact_dir)
        break
    except Exception:
        print("Missing:", artifact_name)

if artifact_dir is None:
    raise RuntimeError("No checkpoint artifact found for this run.")

pt_files = glob.glob(f"{artifact_dir}/**/*.pt", recursive=True)
if not pt_files:
    raise RuntimeError(f"No .pt file found inside artifact {artifact_name_used}")

resume_ckpt = pt_files[0]
print("Chosen artifact resume checkpoint:", resume_ckpt)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


Missing: atin5551-uc-davis/attnres-next-scale/vlm_attnres_flickr30k_t4_step_0004000:latest


wandb: Downloading large artifact 'vlm_attnres_flickr30k_t4_step_0003500:latest', 1408.80MB. 5 files...
wandb:   5 of 5 files downloaded.  
Done. 00:00:17.1 (82.6MB/s)


Downloaded: atin5551-uc-davis/attnres-next-scale/vlm_attnres_flickr30k_t4_step_0003500:latest
To: /content/wandb_restores
Chosen artifact resume checkpoint: /content/wandb_restores/step_0003500.pt


In [18]:
%cd /content/AttnResGPT-next-scale
!python experiments/vlm_attnres_flickr30k.py \
  --entity atin5551-uc-davis \
  --project attnres-next-scale \
  --run-name vlm_attnres_flickr30k_t4_resume_clean \
  --vision-model google/siglip-base-patch16-224 \
  --dataset-name Mozilla/flickr30k-transformed-captions \
  --dataset-split train \
  --tokenizer-name gpt2 \
  --decoder-config configs/large_ctx512_3000.yaml \
  --decoder-architecture attnres \
  --decoder-backend gpt_attnres_large \
  --device cuda \
  --batch-size 8 \
  --max-examples 20000 \
  --max-text-tokens 128 \
  --num-workers 2 \
  --epochs 3 \
  --learning-rate 1e-4 \
  --weight-decay 0.01 \
  --warmup-steps 100 \
  --checkpoint-interval 500 \
  --plateau-patience 2 \
  --mixed-precision \
  --amp-dtype float16 \
  --wandb-resume never \
  --resume-from "$resume_ckpt"


/content/AttnResGPT-next-scale
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100% 408/408 [00:00<00:00, 682.80it/s, Materializing param=vision_model.post_layernorm.weight]                      0:00<00:00, 687.97it/s, Materializing param=vision_model.encoder.layers.4.self_attn.k_proj.weight]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: atin5551 (atin5551-uc-davis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in

In [5]:
!find /content/AttnResGPT-next-scale/checkpoints -maxdepth 2 -type f | sort | tail -50
!find /content/AttnResGPT-next-scale/outputs -maxdepth 3 -type f | grep -E "vlm_|alpha_summary|vision_vs_language|png|json" | sort | tail -100


/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0000500.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0001000.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0001500.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0002000.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0002500.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0003000.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0003500.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0004000.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0004500.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0005000.pt
/content/AttnResGPT-next-scale/checkpoints/vlm_baseline_flickr30k_t4_b8_2/step_0005500.pt
/content/A

In [6]:
!tar -czf /content/vlm_final_backup.tar.gz \
  -C /content/AttnResGPT-next-scale \
  checkpoints \
  outputs
!ls -lh /content/vlm_final_backup.tar.gz


-rw-r--r-- 1 root root 18G Mar 26 22:55 /content/vlm_final_backup.tar.gz


In [7]:
import wandb, os


run = wandb.init(project="attnres-next-scale", entity="atin5551-uc-davis", job_type="final-backup")
artifact = wandb.Artifact("vlm-final-backup", type="backup")
artifact.add_file("/content/vlm_final_backup.tar.gz")
run.log_artifact(artifact)
run.finish()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: atin5551 (atin5551-uc-davis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


OSError: [Errno 28] No space left on device: '/content/vlm_final_backup.tar.gz' -> '/root/.local/share/wandb/artifacts/staging/tmpkis4fjxs'